# Pipeline Check -- dry run of the full GridRoute + MazeBench pipeline (CHECK/TEST notebook)

Companion to `kaggle_train.ipynb`. Runs the **exact same pipeline, same code, same stages, in the same order** -- setup, feasibility check, Phase 1 baselines, Phase 1b full-precision ablation, Phase 2 recipes 1-3 (SFT -> GRPO timing test -> GRPO full run -> eval, x3 conditions), run summary, results aggregation -- just at a tiny scale (2-16 samples/tasks per stage instead of 20-400, 2-3 GRPO steps instead of 20-hundreds) so the whole thing finishes in well under two hours instead of potentially days.

**What this is for:** catching syntax errors, API-incompatibility (TRL/PEFT version drift), OOM, and memory-not-actually-released-between-stages problems *before* committing real GPU-hours to `kaggle_train.ipynb`. It is not a substitute for that notebook -- 2 samples tell you nothing about accuracy, only whether the code runs. Every stage here calls the real, unmodified project scripts (`eval.py`, `train_sft.py`, `train_grpo.py`, `check_finetune_feasibility.py`) with smaller `--n`/`--n_tasks`/`--max_steps` arguments -- nothing is mocked, stubbed, or faked; every model load, generation, training step, and score below is real.

**How to read the result:** scroll to the **"Pipeline check verdict"** cell at the end (or just check whether this Kaggle run committed with a green checkmark or a red error). Unlike the real notebook -- which deliberately never lets one bad stage kill an unattended multi-hour run -- this notebook raises a real exception and fails the whole commit if *any* stage did not complete cleanly, since "did everything work" is this notebook's entire purpose. If it fails, the failed stage(s) are named explicitly in that cell's output, and the actual error is in that stage's own cell output further up.

**Before running:** identical requirements to `kaggle_train.ipynb` -- GPU T4 x2 (or P100), Internet On, a `GITHUB_TOKEN` secret (fine-grained, read-only, toggled ON for this notebook) to clone the private repo, optionally `HF_TOKEN` for faster downloads. See that notebook's intro cell for the full setup walkthrough; not repeated here.

**Expected runtime:** roughly 45-90 minutes on a Kaggle T4, dominated by model download/load time (4 models loaded repeatedly across stages) more than actual compute -- an estimate, not a guarantee. Still an order of magnitude or more less than the real notebook.

Generated from the same `build_notebook.py` as `kaggle_train.ipynb` (`build_cells(check=True)` vs `build_cells(check=False)`) so the two can't structurally drift apart -- if you change the real pipeline's stages, regenerate both, don't hand-edit either `.ipynb` directly.

In [6]:
import os, subprocess, sys

REPO_URL = "github.com/Vedang-P/neuro-symbolic-pathfinding.git"
REPO_DIR = "/kaggle/working/neuro-symbolic-pathfinding"

# GIT_TERMINAL_PROMPT=0: if auth still fails for some other reason (token
# lacks access to this repo, expired, etc.), git fails immediately with a
# clear error instead of hanging on an interactive password prompt that has
# nowhere to go in a notebook environment.
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True, env=env)
elif not os.path.isdir(REPO_DIR):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        # Token must go in the PASSWORD slot (after the colon), not as a bare
        # username before @ -- a bare-username URL leaves git still needing a
        # password, which it then can't prompt for here ("could not read
        # Password ... No such device or address"). "x-access-token" as the
        # username is GitHub's documented convention for token-based HTTPS auth.
        clone_url = f"https://x-access-token:{token}@{REPO_URL}"
        subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True, env=env)
        print("Cloned via GITHUB_TOKEN secret.")
    except Exception as e:
        print(f"Could not clone via secret ({type(e).__name__}: {e}).")
        print("Check: the secret is attached to this notebook (Add-ons -> Secrets -> make sure")
        print("it's toggled ON for this notebook specifically), the token hasn't expired, and it")
        print("has read access to this exact repo (fine-grained tokens need per-repo access grants).")
        print("If you attached the repo as a Kaggle Dataset instead, set REPO_DIR above to its mount")
        print("path (typically /kaggle/input/<dataset-name>) and re-run this cell.")
        raise

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working dir:", os.getcwd())


Repo already present, pulling latest...
Already up to date.
Working dir: /kaggle/working/neuro-symbolic-pathfinding


In [7]:
# Pulls in alphamaze_reference/ (github.com/menloresearch/visual-thinker) -- eval.py uses their
# real MazeBench scoring code directly from this submodule when present, falling back to a less
# faithful exact-match approach (with a loud warning) if it isn't.
subprocess.run(["git", "submodule", "update", "--init"], check=True)


CompletedProcess(args=['git', 'submodule', 'update', '--init'], returncode=0)

In [8]:
# Optional: read from Kaggle Secrets, never pasted here -- higher HF Hub rate
# limits/download speed, and required if any candidate model is gated.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # some older library versions check this name
    print("HF_TOKEN set from Kaggle secret.")
except Exception as e:
    print(f"No HF_TOKEN secret found ({type(e).__name__}) -- proceeding unauthenticated "
          "(fine for these public models, just slower/rate-limited).")


HF_TOKEN set from Kaggle secret.


In [9]:
# requirements.txt pins peft>=0.19.0, which ships its own Gemma4ClippableLinear
# support -- Gemma 4 no longer needs Unsloth at all (see hf_models.py's
# load_trainable_model() docstring), so there's no separate unsloth install here.
%pip install -q -r requirements.txt
# Kaggle's preinstalled wandb has been observed with its SDK and its own
# bundled generated protobuf file out of sync ("cannot import name 'Imports'
# from wandb.proto.wandb_telemetry_pb2"), which crashes on import -- trl's
# base_trainer.py checks wandb availability at import time, unconditionally,
# regardless of Unsloth. We never use wandb (report_to=[] everywhere) --
# removing it outright is the surest fix, on top of WANDB_DISABLED below.
%pip uninstall -q -y wandb


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
# Belt and suspenders alongside hf_models.configure_quiet_logging() (which
# also sets these for every subprocess this notebook launches) -- set here
# too in case anything in the notebook's own kernel process ever imports
# wandb directly.
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Run infrastructure: `run_stage()`

Every pipeline stage below goes through this. It never raises -- a stage that OOMs, hits a bug, or has a transient HF Hub hiccup is logged and skipped over, not fatal to the rest of an unattended run. `required_paths` lets a stage declare "don't even try me if this input is missing" (e.g. a GRPO stage needing the preceding SFT stage's checkpoint), so a failure shows up as a clear, attributable skip reason in the log instead of a cryptic error three layers deep in someone else's library. `steps_for_budget()` reads a prior GRPO timing-test's measured per-step cost and sizes the following full run to fit a wall-clock budget automatically, which is what makes "no human between cells" actually work -- without it, the standard advice ("check the timing printout, adjust --max_steps") requires exactly the babysitting this notebook is built to avoid.

In [11]:
import json
import subprocess
import sys
import time
from pathlib import Path

RESULTS_DIR = Path("./results_check")
STAGE_LOG_PATH = RESULTS_DIR / "stage_log.json"
stage_results = json.loads(STAGE_LOG_PATH.read_text()) if STAGE_LOG_PATH.exists() else []

# Safety-ceiling timeouts (a hard kill-switch if something hangs) -- these are
# NOT the target duration for a stage, just an upper bound. Full GRPO runs
# size their own --max_steps well below their timeout via steps_for_budget().
TIMEOUT_QUICK = 10 * 60        # feasibility check, GRPO timing tests
# This check notebook uses a tiny --n (2) on every benchmark call, so
# even MazeBench's worst-case ~180s/maze (measured on real hardware at 8192
# max_new_tokens, see kaggle_train.ipynb) keeps every eval call well under
# TIMEOUT_EVAL. Not trying to reproduce a meaningful accuracy number here --
# only to prove every stage runs end to end without error.
TIMEOUT_EVAL = 15 * 60         # baseline/checkpoint evals
TIMEOUT_SFT = 20 * 60          # SFT warm-start
TIMEOUT_GRPO_FULL = 20 * 60  # full GRPO training runs, ceiling above their own time budget

# Target wall-clock budget per full GRPO training run -- lower this if your
# weekly quota is tight, raise it if you have room. Applied per condition, so
# the pricier "consistency" condition naturally gets fewer steps than
# "single"/"mixed" for the same budget, without any special-casing.
GRPO_BUDGET_MINUTES = 2


def _save_stage_log():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    STAGE_LOG_PATH.write_text(json.dumps(stage_results, indent=2))


def run_stage(name, args, required_paths=None, timeout=None):
    """Run `python <args>` as a subprocess. Always returns, never raises."""
    print(f"\n{'='*70}\n\u25b6 {name}\n{'='*70}")
    for p in (required_paths or []):
        if not Path(p).exists():
            print(f"\u23ed  SKIPPED -- required input not found: {p}")
            stage_results.append({"stage": name, "status": "skipped", "reason": f"missing {p}"})
            _save_stage_log()
            return stage_results[-1]

    t0 = time.time()
    try:
        proc = subprocess.run([sys.executable] + args, timeout=timeout)
        elapsed_min = round((time.time() - t0) / 60, 1)
        if proc.returncode == 0:
            print(f"\n\u2705 DONE ({elapsed_min} min)")
            result = {"stage": name, "status": "ok", "elapsed_min": elapsed_min}
        else:
            print(f"\n\u274c FAILED (exit code {proc.returncode}, {elapsed_min} min) -- see output above")
            result = {"stage": name, "status": "failed", "exit_code": proc.returncode, "elapsed_min": elapsed_min}
    except subprocess.TimeoutExpired:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u23f1  TIMED OUT after {elapsed_min} min (limit {timeout/60:.0f} min)")
        result = {"stage": name, "status": "timeout", "elapsed_min": elapsed_min}
    except Exception as e:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u274c FAILED (exception: {e})")
        result = {"stage": name, "status": "error", "error": str(e), "elapsed_min": elapsed_min}

    stage_results.append(result)
    _save_stage_log()
    return result


def steps_for_budget(timing_dir, budget_minutes=GRPO_BUDGET_MINUTES, min_steps=3, default_steps=3):
    """Read a prior train_grpo.py run's timing.json and compute a step count
    that fits budget_minutes at that measured per-step cost. Falls back to
    default_steps if timing.json is missing (e.g. the timing test itself
    failed) -- better to attempt a bounded full run than skip it outright."""
    timing_path = Path(timing_dir) / "timing.json"
    if not timing_path.exists():
        print(f"  (no timing.json at {timing_dir}, using default {default_steps} steps)")
        return default_steps
    t = json.loads(timing_path.read_text())
    per_step = t.get("per_step_s", 0)
    if per_step <= 0:
        return default_steps
    steps = max(min_steps, int(budget_minutes * 60 / per_step))
    print(f"  (timing.json: {per_step:.1f}s/step -> {steps} steps fits a {budget_minutes}min budget)")
    return steps


## Feasibility check: does Gemma 4 E2B/E4B LoRA actually fit this GPU?

Don't assume the ~8-10GB (E2B) / ~17GB (E4B) numbers from elsewhere -- confirm on this exact hardware.

In [12]:
run_stage("Feasibility check (all candidate models)", ["check_finetune_feasibility.py"], timeout=TIMEOUT_QUICK)



▶ Feasibility check (all candidate models)
GPU: Tesla T4
Total VRAM: 15.64 GB
CUDA allocated before tests: 0.00 GB

Testing: deepseek-r1-distill-qwen-1.5b  (deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B)
  Loading + attaching LoRA (r=16)...


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 222.85it/s]


  Backend: bnb_peft
  Running forward+backward (seq_len<=256, batch=1)...

  VRAM breakdown (summed across 2 GPU(s)):
    Before:       0.00 GB
    After LoRA:   2.62 GB  (+2.62)
    After fwd+bwd:2.72 GB
    Peak:         3.01 GB
    Est. GRPO:    4.21 GB  → ✅ FEASIBLE  (rough estimate, see docstring)

Testing: smollm2-1.7b  (HuggingFaceTB/SmolLM2-1.7B-Instruct)
  Loading + attaching LoRA (r=16)...


Loading weights: 100%|██████████| 218/218 [00:01<00:00, 167.96it/s]


  Backend: bnb_peft
  Running forward+backward (seq_len<=256, batch=1)...

  VRAM breakdown (summed across 2 GPU(s)):
    Before:       0.70 GB
    After LoRA:   2.00 GB  (+1.31)
    After fwd+bwd:2.08 GB
    Peak:         2.13 GB
    Est. GRPO:    2.98 GB  → ✅ FEASIBLE  (rough estimate, see docstring)

Testing: gemma4-e2b  (unsloth/gemma-4-E2B-it)
  Loading + attaching LoRA (r=16)...


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 528.40it/s] 


  Backend: bnb_peft
  Running forward+backward (seq_len<=256, batch=1)...

  VRAM breakdown (summed across 2 GPU(s)):
    Before:       0.85 GB
    After LoRA:   6.77 GB  (+5.92)
    After fwd+bwd:6.79 GB
    Peak:         6.86 GB
    Est. GRPO:    9.60 GB  → ✅ FEASIBLE  (rough estimate, see docstring)

Testing: gemma4-e4b  (unsloth/gemma-4-E4B-it)
  Loading + attaching LoRA (r=16)...


Loading weights: 100%|██████████| 2076/2076 [00:40<00:00, 50.67it/s] 


  Backend: bnb_peft
  Running forward+backward (seq_len<=256, batch=1)...

  VRAM breakdown (summed across 2 GPU(s)):
    Before:       0.99 GB
    After LoRA:   9.34 GB  (+8.35)
    After fwd+bwd:9.37 GB
    Peak:         9.43 GB
    Est. GRPO:    13.20 GB  → ✅ FEASIBLE  (rough estimate, see docstring)

MODEL                    BACKEND     LOAD+LORA   PEAK   GRPO  VERDICT
--------------------------------------------------------------------------------
deepseek-r1-distill-qwen-1.5b bnb_peft         2.6G   3.0G   4.2G  ✅ FEASIBLE
smollm2-1.7b             bnb_peft         1.3G   2.1G   3.0G  ✅ FEASIBLE
gemma4-e2b               bnb_peft         5.9G   6.9G   9.6G  ✅ FEASIBLE
gemma4-e4b               bnb_peft         8.3G   9.4G  13.2G  ✅ FEASIBLE

✅ DONE (3.8 min)


{'stage': 'Feasibility check (all candidate models)',
 'status': 'ok',
 'elapsed_min': 3.8}

## Phase 1: baselines -- Gemma 4 E2B/E4B + AlphaMaze on MazeBench and GridRoute 5x5

AlphaMaze should land near its published 93% on MazeBench (using their real scoring code via the submodule) -- this is the harness sanity check. Gemma 4's numbers on both benchmarks are the actual open question: no SLM has been tested on GridRoute anywhere in the literature found so far.

(This notebook uses n=2 everywhere below -- too few samples for the 93% figure to mean anything statistically. The point here is only proving the harness runs; the real replication check lives in `kaggle_train.ipynb`.)

In [13]:
ALPHAMAZE_LOCAL_PATH = "data/models/alphamaze-v0.2-1.5b"
os.makedirs("data/models", exist_ok=True)
try:
    if not os.path.isdir(ALPHAMAZE_LOCAL_PATH):
        from huggingface_hub import snapshot_download
        snapshot_download("homebrewltd/AlphaMaze-v0.2-1.5B", local_dir=ALPHAMAZE_LOCAL_PATH)
    print("AlphaMaze checkpoint ready.")
except Exception as e:
    print(f"\u26a0\ufe0f  AlphaMaze checkpoint download failed ({e}) -- AlphaMaze-specific "
          "stages below will be skipped (they require-path-check this directory).")


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

AlphaMaze checkpoint ready.


This cell exercises the exact same AlphaMaze-on-MazeBench path as `kaggle_train.ipynb`'s Phase 1 replication check (full precision, real submodule scoring via `alphamaze_reference`) but with only 2 mazes and the standard 4096-token floor instead of 16384 -- not trying to reproduce the published ~93% here, only to prove the checkpoint loads, generates, streams, and scores correctly end to end. The real replication check lives in `kaggle_train.ipynb`.

In [14]:
run_stage("AlphaMaze baseline: MazeBench (full precision, pipeline check)",
          ["eval.py", "--model", "alphamaze", "--benchmark", "mazebench", "--n", "2",
           "--no_4bit", "--output_dir", "./results_check/eval"],
          required_paths=[ALPHAMAZE_LOCAL_PATH], timeout=15 * 60)
run_stage("AlphaMaze baseline: GridRoute NL",
          ["eval.py", "--model", "alphamaze", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[ALPHAMAZE_LOCAL_PATH], timeout=TIMEOUT_EVAL)



▶ AlphaMaze baseline: MazeBench (full precision, pipeline check)
Model: ./data/models/alphamaze-v0.2-1.5b
4bit=False  backend=hf  benchmark=mazebench


MazeBench:   0%|          | 0/2 [00:00<?, ?maze/s]

  [1/2] generating (streamed below)...
<think>
Step 1: Go left
<|0-0|><|up_down_left_wall|><|blank|><|0-1|><|up_down_wall|><|blank|><|0-2|><|up_down_wall|><|blank|><|0-3|><|up_down_wall|><|blank|><|0-4|><|up_right_wall|><|blank|>
<|1-0|><|up_left_wall|><|blank|><|1-1|><|up_down_wall|><|blank|><|1-2|><|up_down_wall|><|left|><|1-3|><|up_right_wall|><|blank|><|1-4|><|left_right_wall|><|blank|>
<|2-0|><|down_left_wall|><|blank|><|2-1|><|up_right_wall|><|blank|><|2-2|><|up_left_wall|><|blank|><|2-3|><|down_right_wall|><|blank|><|2-4|><|left_right_wall|><|blank|>
<|3-0|><|up_left_right_wall|><|blank|><|3-1|><|left_right_wall|><|blank|><|3-2|><|down_left_wall|><|blank|><|3-3|><|up_down_wall|><|blank|><|3-4|><|down_right_wall|><|blank|>
<|4-0|><|down_left_wall|><|blank|><|4-1|><|down_wall|><|blank|><|4-2|><|up_down_wall|><|blank|><|4-3|><|up_down_wall|><|target|><|4-4|><|up_down_right_wall|><|blank|>

Step 2: Go left
<|0-0|><|up_down_left_wall|><|blank|><|0-1|><|up_down_wall|><|blank|><|0-2|><

MazeBench:  50%|█████     | 1/2 [00:30<00:30, 30.59s/maze, acc=1/1]


  [1/2] ok  level=medium pred_len=8 gt_len=8
  [2/2] generating (streamed below)...
<think>
Step 1: Go down
<|0-0|><|up_left_right_wall|><|blank|><|0-1|><|up_left_wall|><|blank|><|0-2|><|up_right_wall|><|blank|><|0-3|><|up_left_wall|><|blank|><|0-4|><|up_right_wall|><|target|>
<|1-0|><|left_wall|><|blank|><|1-1|><|down_right_wall|><|blank|><|1-2|><|down_left_wall|><|blank|><|1-3|><|down_right_wall|><|blank|><|1-4|><|left_right_wall|><|blank|>
<|2-0|><|left_right_wall|><|blank|><|2-1|><|up_left_wall|><|down|><|2-2|><|up_down_wall|><|blank|><|2-3|><|up_down_wall|><|blank|><|2-4|><|down_right_wall|><|blank|>
<|3-0|><|left_right_wall|><|blank|><|3-1|><|down_left_right_wall|><|blank|><|3-2|><|up_down_left_wall|><|blank|><|3-3|><|up_down_wall|><|blank|><|3-4|><|up_right_wall|><|blank|>
<|4-0|><|down_left_wall|><|blank|><|4-1|><|up_down_wall|><|blank|><|4-2|><|up_down_wall|><|blank|><|4-3|><|up_down_wall|><|blank|><|4-4|><|down_right_wall|><|blank|>
Hit a dead end.
RESET.
Step 1: Go right
<|

...

GridRoute-nl-5x5:  50%|█████     | 1/2 [00:19<00:19, 19.78s/task, optimal=0/1, valid=0/1]


  [1/2] INVAL start=(0, 4) goal=(4, 1) opt_len=7
  [2/2] generating (streamed below)...
<think>
First, I need to navigate from the starting position (2,0) to the target (4,3) on a 5x5 grid. The grid has an obstacle at (0,1), which I must avoid.

I'll start by moving right to clear the path. This takes me to (2,1), but since (0,1) is an obstacle, I can't move left from there. So, I'll move up to (2,2).

Next, I'll move up to (2,3), then up again to (2,4). Now, I can move left to (1,4). From here, I'll move left to (0,4) and then up to (0,3). 

Finally, I'll move right to (1,3), then right to (2,3), and up to (3,3), and up to (4,3) to reach the target.
</think>

To navigate from (2,0) to (4,3) on a 5x5 grid with an obstacle at (0,1), follow these steps:

1. **Move up** to (2,1) (but this is blocked by the obstacle, so adjust accordingly).
2. **Move up** to (2,2).
3. **Move up** to (2,3).
4. **Move up** to (2,4).
5. **Move left** to (1,4).
6. **Move left** to (0,4).
7. **Move up** to (0,

GridRoute-nl-5x5: 100%|██████████| 2/2 [00:38<00:00, 19.41s/task, optimal=0/2, valid=0/2]



✅ DONE (0.9 min)


{'stage': 'AlphaMaze baseline: GridRoute NL',
 'status': 'ok',
 'elapsed_min': 0.9}

In [15]:
run_stage("Gemma 4 E2B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "mazebench", "--n", "2",
           "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)
run_stage("Gemma 4 E2B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)



▶ Gemma 4 E2B baseline: MazeBench
Model: unsloth/gemma-4-E2B-it
4bit=True  backend=hf  benchmark=mazebench


MazeBench:   0%|          | 0/2 [00:00<?, ?maze/s]

  [1/2] generating (streamed below)...
thought
The user wants me to solve a maze.
I need to track the current position (row, col) and the sequence of moves.
The maze is represented by tokens.

Maze structure analysis:
The maze is a grid. Let's define coordinates (row, col).

Maze representation:
Row 0: <|0-0|><|up_down_left_wall|><|blank|><|0-1|><|up_down_wall|><|blank|><|0-2|><|up_down_wall|><|blank|><|0-3|><|up_down_wall|><|blank|><|0-4|><|up_right_wall|><|blank|>
Row 1: <|1-0|><|up_left_wall|><|blank|><|1-1|><|up_down_wall|><|blank|><|1-2|><|up_down_wall|><|origin|><|1-3|><|up_right_wall|><|blank|><|1-4|><|left_right_wall|><|blank|>
Row 2: <|2-0|><|down_left_wall|><|blank|><|2-1|><|up_right_wall|><|blank|><|2-2|><|up_left_wall|><|blank|><|2-3|><|down_right_wall|><|blank|><|2-4|><|left_right_wall|><|blank|>
Row 3: <|3-0|><|up_left_right_wall|><|blank|><|3-1|><|left_right_wall|><|blank|><|3-2|><|down_left_wall|><|blank|><|3-3|><|up_down_wall|><|blank|><|3-4|><|down_right_wall|><|blank

MazeBench:  50%|█████     | 1/2 [08:48<08:48, 528.93s/maze, acc=0/1]


  [1/2] no-answer level=medium pred_len=0 gt_len=8
  [2/2] generating (streamed below)...
thought
The user wants me to solve a maze.
I need to track the current position (row, col) and the sequence of moves.
The maze is represented by tokens.

Maze structure analysis:
The maze is a grid. Let's define coordinates (row, col).

Maze representation:
Row 0: <|0-0|><|up_left_right_wall|><|blank|><|0-1|><|up_left_wall|><|blank|><|0-2|><|up_right_wall|><|blank|><|0-3|><|up_left_wall|><|blank|><|0-4|><|up_right_wall|><|target|>
Row 1: <|1-0|><|left_wall|><|blank|><|1-1|><|down_right_wall|><|blank|><|1-2|><|down_left_wall|><|blank|><|1-3|><|down_right_wall|><|blank|><|1-4|><|left_right_wall|><|blank|>
Row 2: <|2-0|><|left_right_wall|><|blank|><|2-1|><|up_left_wall|><|origin|><|2-2|><|up_down_wall|><|blank|><|2-3|><|up_down_wall|><|blank|><|2-4|><|down_right_wall|><|blank|>
Row 3: <|3-0|><|left_right_wall|><|blank|><|3-1|><|down_left_right_wall|><|blank|><|3-2|><|up_down_left_wall|><|blank|><|3-

...

GridRoute-nl-5x5:  50%|█████     | 1/2 [02:50<02:50, 170.50s/task, optimal=1/1, valid=1/1]


  [1/2] OPT   start=(0, 4) goal=(4, 1) opt_len=7
  [2/2] generating (streamed below)...
thought
The user wants me to find a path from the starting point (2,0) to the target point (4,3) on a 5x5 grid.
The grid coordinates are (row, col), where rows are 0 to 4 and columns are 0 to 4.
The obstacles are at [(0,1)].
Allowed moves are up, down, left, right.

Start: S = (2, 0)
Target: T = (4, 3)
Grid size: 5x5.

Obstacles: O = {(0, 1)}

I need to find a sequence of valid moves. Since the problem doesn't specify if I need the shortest path, I will aim for a simple, valid path.

Let's represent the grid and track visited cells to avoid loops, although for a simple path finding, just ensuring validity is key.

Grid visualization (R, C):
(0,0) (0,1)X (0,2) (0,3) (0,4)
(1,0) (1,1) (1,2) (1,3) (1,4)
(2,0) (2,1) (2,2) (2,3) (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3) (4,4)

Start (2, 0). Target (4, 3).

Path planning:
1. From (2, 0). Possible moves: (1, 0), (3, 0), (2, 1).
2. Let's 

GridRoute-nl-5x5: 100%|██████████| 2/2 [05:30<00:00, 165.21s/task, optimal=2/2, valid=2/2]



✅ DONE (5.9 min)


{'stage': 'Gemma 4 E2B baseline: GridRoute NL',
 'status': 'ok',
 'elapsed_min': 5.9}

E4B baseline (inference-only, no LoRA) is much lighter than E4B *training* -- run this regardless of what the feasibility check said about training, since it doesn't need LoRA attachment at all, just enough VRAM to hold the weights at 4-bit.

In [16]:
run_stage("Gemma 4 E4B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "mazebench", "--n", "2",
           "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)
run_stage("Gemma 4 E4B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)



▶ Gemma 4 E4B baseline: MazeBench
Model: unsloth/gemma-4-E4B-it
4bit=True  backend=hf  benchmark=mazebench


MazeBench:   0%|          | 0/2 [00:00<?, ?maze/s]

  [1/2] generating (streamed below)...
thought
The user wants me to solve a maze represented by a series of tokens.
I need to find the sequence of movements (<|up|>, <|down|>, <|left|>, <|right|>) from the origin to the target.
I must output only the move tokens, separated by spaces.

The maze structure is a grid where each cell contains information about its coordinates, walls, and connectivity.

Let's analyze the maze structure. It seems to be a 5x5 grid of cells, where each cell is represented by a token sequence.

The maze is given as:
Row 0: <|0-0|><|up_down_left_wall|><|blank|><|0-1|><|up_down_wall|><|blank|><|0-2|><|up_down_wall|><|blank|><|0-3|><|up_down_wall|><|blank|><|0-4|><|up_right_wall|><|blank|>
Row 1: <|1-0|><|up_left_wall|><|blank|><|1-1|><|up_down_wall|><|blank|><|1-2|><|up_down_wall|><|origin|><|1-3|><|up_right_wall|><|blank|><|1-4|><|left_right_wall|><|blank|>
Row 2: <|2-0|><|down_left_wall|><|blank|><|2-1|><|up_right_wall|><|blank|><|2-2|><|up_left_wall|><|blank|><

MazeBench:  50%|█████     | 1/2 [11:24<11:24, 684.09s/maze, acc=0/1]


  [1/2] no-answer level=medium pred_len=0 gt_len=8
  [2/2] generating (streamed below)...
thought
The user wants me to solve a maze represented by a series of tokens.
I need to find the sequence of movements (<|up|>, <|down|>, <|left|>, <|right|>) from the origin to the target.
I must output only the move tokens, separated by spaces.

The maze structure is a grid of cells, where each cell contains information about its coordinates, walls, and content (blank, origin, target).

Let's analyze the maze structure. It seems to be a 5x5 grid (5 rows, 5 columns).

Row 0: <|0-0|> <|up_left_right_wall|> <|blank|> <|0-1|> <|up_left_wall|> <|blank|> <|0-2|> <|up_right_wall|> <|blank|> <|0-3|> <|up_left_wall|> <|blank|> <|0-4|> <|up_right_wall|> <|target|>
Wait, the input format is a continuous stream of tokens. I need to parse it into a grid structure.

The input is:
MAZE: <|0-0|><|up_left_right_wall|><|blank|><|0-1|><|up_left_wall|><|blank|><|0-2|><|up_right_wall|><|blank|><|0-3|><|up_left_wall|

GridRoute-nl-5x5:   0%|          | 0/2 [00:00<?, ?task/s]

  [1/2] generating (streamed below)...
thought
The user wants me to find a path from the starting point (0, 4) to the goal point (4, 1) on a 5x5 grid.
The grid coordinates are (row, column), where rows range from 0 to 4 and columns range from 0 to 4.
The grid size is 5x5.
There is one obstacle at (3, 2).
Movement is restricted to up, down, left, or right (no diagonals).

Start: S = (0, 4)
Goal: G = (4, 1)
Obstacle: O = (3, 2)

I will use Breadth-First Search (BFS) or a similar pathfinding algorithm to find the shortest path.

Grid visualization (R, C):
(0,0) (0,1) (0,2) (0,3) (0,4)
(1,0) (1,1) (1,2) (1,3) (1,4)
(2,0) (2,1) (2,2) (2,3) (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3) (4,4)

Obstacle O = (3, 2).

Let's perform BFS.

Queue: [(start_node, path)]
Visited: set()

1. Initialize:
Queue = [((0, 4), [(0, 4)])]
Visited = {(0, 4)}

2. Dequeue ((0, 4), [(0, 4)]). Neighbors of (0, 4):
   (0, 3) - Valid
   (1, 4) - Valid
   (0, 5) - Out of bounds
   (-1, 4) - Out of bounds

GridRoute-nl-5x5:  50%|█████     | 1/2 [10:54<10:54, 654.90s/task, optimal=1/1, valid=1/1]


  [1/2] OPT   start=(0, 4) goal=(4, 1) opt_len=7
  [2/2] generating (streamed below)...
thought
The user wants me to find a path from the starting point (2, 0) to the goal point (4, 3) on a 5x5 grid.
The allowed moves are up, down, left, right.
There is one obstacle at (0, 1).

The grid size is 5x5. I will assume coordinates are (row, column), where rows and columns range from 0 to 4.

Start: S = (2, 0)
Goal: G = (4, 3)
Obstacle: O = (0, 1)

Grid visualization (R x C):
(0,0) (0,1) (0,2) (0,3) (0,4)
(1,0) (1,1) (1,2) (1,3) (1,4)
(2,0) (2,1) (2,2) (2,3) (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3)

Start (2, 0) is safe.
Goal (4, 3) is safe.
Obstacle (0, 1) is safe.

I will use Breadth-First Search (BFS) or a similar pathfinding algorithm to find the shortest path.

**BFS Implementation:**

Queue: [(current_position, path)]
Visited: set of positions already checked

1. Initialize:
   Queue = [((2, 0), [(2, 0)])]
   Visited = {(2, 0)}

2. Iteration 1:
   Dequeue: ((2, 0), [

{'stage': 'Gemma 4 E4B baseline: GridRoute NL',
 'status': 'timeout',
 'elapsed_min': 15.0}

## Phase 1b: full-precision ablation for Gemma 4 (how much does 4-bit cost?)

Gemma 4's baselines above ran 4-bit-quantized -- necessary for *training* (LoRA/GRPO need the VRAM headroom), but baseline inference has no such requirement, so it's worth checking directly rather than assuming the gap is small. Same `--n`/seed as the corresponding 4-bit cell above, so this is a paired comparison on identical tasks, not just a same-average different-sample one. `eval.py` records `load_in_4bit` in its output JSON specifically so the aggregation table below can show both side by side. Gemma 4 E4B at full precision is the one most likely to not fit the T4 at all (4-bit exists exactly to avoid this) -- if that cell fails, that itself is a useful, reportable data point, not a notebook bug.

AlphaMaze isn't part of this ablation at all: it always runs full precision, enforced directly in `eval.py` regardless of any `--load_in_4bit` flag, not just by convention in this notebook. It's the one fixed reference point every other number here gets compared against and it's never trained further in Phase 2, so there's no training-time VRAM constraint forcing a quantization tradeoff the way there is for Gemma 4 -- no reason to ever accept the noise 4-bit would add to its numbers.

In [17]:
run_stage("[full-precision] Gemma 4 E2B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "mazebench", "--n", "2",
           "--no_4bit", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)
run_stage("[full-precision] Gemma 4 E2B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--no_4bit", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)



▶ [full-precision] Gemma 4 E2B baseline: MazeBench
Model: unsloth/gemma-4-E2B-it
4bit=False  backend=hf  benchmark=mazebench


MazeBench:   0%|          | 0/2 [00:00<?, ?maze/s]

  [1/2] generating (streamed below)...
thought
The user wants me to solve a maze and output the sequence of movements from the origin to the target.

The maze is represented by a series of tokens. I need to parse this into a grid structure and find a path.

**Maze Parsing:**

The maze is given in rows. Let's define the dimensions.
The coordinates seem to be (row-col).

Row 0: <|0-0|> <|up_down_left_wall|> <|blank|> <|0-1|> <|up_down_wall|> <|blank|> <|0-2|> <|up_down_wall|> <|blank|> <|0-3|> <|up_down_wall|> <|blank|> <|0-4|> <|up_right_wall|> <|blank|>
Row 1: <|1-0|> <|up_left_wall|> <|blank|> <|1-1|> <|up_down_wall|> <|blank|> <|1-2|> <|up_down_wall|> <|origin|> <|1-3|> <|up_right_wall|> <|blank|> <|1-4|> <|left_right_wall|> <|blank|>
Row 2: <|2-0|> <|down_left_wall|> <|blank|> <|2-1|> <|up_right_wall|> <|blank|> <|2-2|> <|up_left_wall|> <|blank|> <|2-3|> <|down_right_wall|> <|blank|> <|2-4|> <|left_right_wall|> <|blank|>
Row 3: <|3-0|> <|up_left_right_wall|> <|blank|> <|3-1|> <|left

MazeBench:  50%|█████     | 1/2 [05:40<05:40, 340.03s/maze, acc=0/1]


  [1/2] no-answer level=medium pred_len=0 gt_len=8
  [2/2] generating (streamed below)...
thought
The user wants me to solve a maze and output the sequence of movements from the origin to the target.

The maze is represented by a series of tokens. I need to parse this into a grid structure and find a path.

**Maze Parsing:**

The maze is given as:
R0: <|0-0|> <|up_left_right_wall|> <|blank|> <|0-1|> <|up_left_wall|> <|blank|> <|0-2|> <|up_right_wall|> <|blank|> <|0-3|> <|up_left_wall|> <|blank|> <|0-4|> <|up_right_wall|> <|target|>
R1: <|1-0|> <|left_wall|> <|blank|> <|1-1|> <|down_right_wall|> <|blank|> <|1-2|> <|down_left_wall|> <|blank|> <|1-3|> <|down_right_wall|> <|blank|> <|1-4|> <|left_right_wall|> <|blank|>
R2: <|2-0|> <|left_right_wall|> <|blank|> <|2-1|> <|up_left_wall|> <|origin|> <|2-2|> <|up_down_wall|> <|blank|> <|2-3|> <|up_down_wall|> <|blank|> <|2-4|> <|down_right_wall|> <|blank|>
R3: <|3-0|> <|left_right_wall|> <|blank|> <|3-1|> <|down_left_right_wall|> <|blank|> <|3

...

GridRoute-nl-5x5:  50%|█████     | 1/2 [03:42<03:42, 222.23s/task, optimal=1/1, valid=1/1]


  [1/2] OPT   start=(0, 4) goal=(4, 1) opt_len=7
  [2/2] generating (streamed below)...
thought
The user wants me to find a path from a starting point $S=(2,0)$ to a target point $T=(4,3)$ on a 5x5 grid.
The grid coordinates are (row, column), typically indexed from (0,0) to (4,4).
The obstacles are at $O = \{(0,1)\}$.
Allowed moves are up, down, left, right (4-connectivity).

Grid size: 5x5. Rows 0-4, Columns 0-4.
Start $S = (2, 0)$.
Target $T = (4, 3)$.
Obstacle $O = \{(0, 1)\}$.

I will use Breadth-First Search (BFS) or simply try to find a path by exploring neighbors, keeping track of visited cells.

Grid visualization (R, C):
(0,0) (0,1)X (0,2) (0,3) (0,4)
(1,0) (1,1) (1,2) (1,3) (1,4)
(2,0)S (2,1) (2,2) (2,3) (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3)T (4,4)

Start: (2, 0)
Target: (4, 3)

Step 1: Initialize queue with S. Visited set initialized with S.
Queue: [(2, 0)]

Step 2: Explore neighbors of (2, 0).
Neighbors:
(1, 0) - Valid. Add to queue.
(3, 0) - Valid. 

GridRoute-nl-5x5: 100%|██████████| 2/2 [06:59<00:00, 209.58s/task, optimal=2/2, valid=2/2]



✅ DONE (7.3 min)


{'stage': '[full-precision] Gemma 4 E2B baseline: GridRoute NL',
 'status': 'ok',
 'elapsed_min': 7.3}

In [18]:
run_stage("[full-precision] Gemma 4 E4B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "mazebench", "--n", "2",
           "--no_4bit", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)
run_stage("[full-precision] Gemma 4 E4B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--no_4bit", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)



▶ [full-precision] Gemma 4 E4B baseline: MazeBench
Model: unsloth/gemma-4-E4B-it
4bit=False  backend=hf  benchmark=mazebench


Loading weights:  68%|██████▊   | 1416/2076 [00:17<00:08, 81.79it/s] 
Traceback (most recent call last):
  File "/kaggle/working/neuro-symbolic-pathfinding/eval.py", line 329, in <module>
    main()
  File "/kaggle/working/neuro-symbolic-pathfinding/eval.py", line 298, in main
    adapter_path=args.checkpoint or None).load()
                                          ^^^^^^
  File "/kaggle/working/neuro-symbolic-pathfinding/hf_models.py", line 263, in load
    self.model = AutoModelForCausalLM.from_pretrained(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 402, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4456, in from_pretrained
    loading_info, disk_offload_index = cls._load_pretrained_model(model, state_dict, checkpoint_files, load_config)
   


❌ FAILED (exit code 1, 0.5 min) -- see output above

▶ [full-precision] Gemma 4 E4B baseline: GridRoute NL
Model: unsloth/gemma-4-E4B-it
4bit=False  backend=hf  benchmark=gridroute-nl


Loading weights:  68%|██████▊   | 1416/2076 [00:04<00:02, 297.75it/s]
Traceback (most recent call last):
  File "/kaggle/working/neuro-symbolic-pathfinding/eval.py", line 329, in <module>
    main()
  File "/kaggle/working/neuro-symbolic-pathfinding/eval.py", line 298, in main
    adapter_path=args.checkpoint or None).load()
                                          ^^^^^^
  File "/kaggle/working/neuro-symbolic-pathfinding/hf_models.py", line 263, in load
    self.model = AutoModelForCausalLM.from_pretrained(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 402, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4456, in from_pretrained
    loading_info, disk_offload_index = cls._load_pretrained_model(model, state_dict, checkpoint_files, load_config)
   


❌ FAILED (exit code 1, 0.4 min) -- see output above


{'stage': '[full-precision] Gemma 4 E4B baseline: GridRoute NL',
 'status': 'failed',
 'exit_code': 1,
 'elapsed_min': 0.4}

## Phase 2, recipe 1: single-format (GridRoute NL only) on Gemma 4 E2B

SFT warm-start, a short GRPO timing test, then a full run sized to fit `GRPO_BUDGET_MINUTES` at the timing test's measured per-step cost -- see the run-infrastructure cell above for why this is computed rather than manually tuned.

In [19]:
SFT_SINGLE_DIR = "./results_check/sft_gemma4-e2b_single"
run_stage("SFT: single-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "nl", "--grid_size", "5",
           "--n_tasks", "16", "--epochs", "1", "--output_dir", SFT_SINGLE_DIR], timeout=TIMEOUT_SFT)



▶ SFT: single-format (Gemma 4 E2B)
Model: unsloth/gemma-4-E2B-it
Format: nl  Tasks: 16  Grid: 5x5  Epochs: 1  LR: 1e-05


Loading weights: 100%|██████████| 1951/1951 [00:42<00:00, 45.90it/s] 


Model + LoRA ready (backend=bnb_peft)
Building SFT dataset (nl, 16 tasks, 5x5)...
  16 training rows.


Map: 100%|██████████| 16/16 [00:00<00:00, 642.88 examples/s]
Traceback (most recent call last):
  File "/kaggle/working/neuro-symbolic-pathfinding/train_sft.py", line 181, in <module>
    main()
  File "/kaggle/working/neuro-symbolic-pathfinding/train_sft.py", line 149, in main
    training_args = SFTConfig(
                    ^^^^^^^^^^
TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'



❌ FAILED (exit code 1, 1.1 min) -- see output above


{'stage': 'SFT: single-format (Gemma 4 E2B)',
 'status': 'failed',
 'exit_code': 1,
 'elapsed_min': 1.1}

In [20]:
GRPO_SINGLE_TIMING_DIR = "./results_check/grpo_gemma4-e2b_single_timing"
run_stage("GRPO timing test: single-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "8", "--max_steps", "2", "--max_completion_length", "1024",
           "--output_dir", GRPO_SINGLE_TIMING_DIR],
          required_paths=[SFT_SINGLE_DIR], timeout=TIMEOUT_QUICK)



▶ GRPO timing test: single-format
⏭  SKIPPED -- required input not found: ./results_check/sft_gemma4-e2b_single


{'stage': 'GRPO timing test: single-format',
 'status': 'skipped',
 'reason': 'missing ./results_check/sft_gemma4-e2b_single'}

In [21]:
GRPO_SINGLE_DIR = "./results_check/grpo_gemma4-e2b_single"
single_steps = steps_for_budget(GRPO_SINGLE_TIMING_DIR)
run_stage(f"GRPO: single-format full run ({single_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "8",
           "--max_steps", str(single_steps), "--max_completion_length", "1024", "--output_dir", GRPO_SINGLE_DIR],
          required_paths=[SFT_SINGLE_DIR], timeout=TIMEOUT_GRPO_FULL)


  (no timing.json at ./results_check/grpo_gemma4-e2b_single_timing, using default 3 steps)

▶ GRPO: single-format full run (3 steps)
⏭  SKIPPED -- required input not found: ./results_check/sft_gemma4-e2b_single


{'stage': 'GRPO: single-format full run (3 steps)',
 'status': 'skipped',
 'reason': 'missing ./results_check/sft_gemma4-e2b_single'}

In [22]:
run_stage("Eval single-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "mazebench", "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_SINGLE_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval single-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_SINGLE_DIR], timeout=TIMEOUT_EVAL)



▶ Eval single-format checkpoint: MazeBench
⏭  SKIPPED -- required input not found: ./results_check/grpo_gemma4-e2b_single

▶ Eval single-format checkpoint: GridRoute NL
⏭  SKIPPED -- required input not found: ./results_check/grpo_gemma4-e2b_single


{'stage': 'Eval single-format checkpoint: GridRoute NL',
 'status': 'skipped',
 'reason': 'missing ./results_check/grpo_gemma4-e2b_single'}

## Phase 2, recipe 2: mixed-format (NL + token, naive)

Does training on both formats (interleaved, same underlying grids) do better than single-format alone -- for either benchmark?

In [23]:
SFT_MIXED_DIR = "./results_check/sft_gemma4-e2b_mixed"
run_stage("SFT: mixed-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "mixed", "--grid_size", "5",
           "--n_tasks", "16", "--epochs", "1", "--output_dir", SFT_MIXED_DIR], timeout=TIMEOUT_SFT)



▶ SFT: mixed-format (Gemma 4 E2B)
Model: unsloth/gemma-4-E2B-it
Format: mixed  Tasks: 16  Grid: 5x5  Epochs: 1  LR: 1e-05


Loading weights: 100%|██████████| 1951/1951 [00:04<00:00, 429.46it/s] 


Model + LoRA ready (backend=bnb_peft)
Building SFT dataset (mixed, 16 tasks, 5x5)...
  32 training rows.


Map: 100%|██████████| 32/32 [00:00<00:00, 7783.45 examples/s]
Traceback (most recent call last):
  File "/kaggle/working/neuro-symbolic-pathfinding/train_sft.py", line 181, in <module>
    main()
  File "/kaggle/working/neuro-symbolic-pathfinding/train_sft.py", line 149, in main
    training_args = SFTConfig(
                    ^^^^^^^^^^
TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'



❌ FAILED (exit code 1, 0.6 min) -- see output above


{'stage': 'SFT: mixed-format (Gemma 4 E2B)',
 'status': 'failed',
 'exit_code': 1,
 'elapsed_min': 0.6}

In [24]:
GRPO_MIXED_TIMING_DIR = "./results_check/grpo_gemma4-e2b_mixed_timing"
run_stage("GRPO timing test: mixed-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "8", "--max_steps", "2", "--max_completion_length", "1024",
           "--output_dir", GRPO_MIXED_TIMING_DIR],
          required_paths=[SFT_MIXED_DIR], timeout=TIMEOUT_QUICK)



▶ GRPO timing test: mixed-format
⏭  SKIPPED -- required input not found: ./results_check/sft_gemma4-e2b_mixed


{'stage': 'GRPO timing test: mixed-format',
 'status': 'skipped',
 'reason': 'missing ./results_check/sft_gemma4-e2b_mixed'}

In [25]:
GRPO_MIXED_DIR = "./results_check/grpo_gemma4-e2b_mixed"
mixed_steps = steps_for_budget(GRPO_MIXED_TIMING_DIR)
run_stage(f"GRPO: mixed-format full run ({mixed_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "8",
           "--max_steps", str(mixed_steps), "--max_completion_length", "1024", "--output_dir", GRPO_MIXED_DIR],
          required_paths=[SFT_MIXED_DIR], timeout=TIMEOUT_GRPO_FULL)


  (no timing.json at ./results_check/grpo_gemma4-e2b_mixed_timing, using default 3 steps)

▶ GRPO: mixed-format full run (3 steps)
⏭  SKIPPED -- required input not found: ./results_check/sft_gemma4-e2b_mixed


{'stage': 'GRPO: mixed-format full run (3 steps)',
 'status': 'skipped',
 'reason': 'missing ./results_check/sft_gemma4-e2b_mixed'}

In [26]:
run_stage("Eval mixed-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "mazebench", "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_MIXED_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval mixed-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_MIXED_DIR], timeout=TIMEOUT_EVAL)



▶ Eval mixed-format checkpoint: MazeBench
⏭  SKIPPED -- required input not found: ./results_check/grpo_gemma4-e2b_mixed

▶ Eval mixed-format checkpoint: GridRoute NL
⏭  SKIPPED -- required input not found: ./results_check/grpo_gemma4-e2b_mixed


{'stage': 'Eval mixed-format checkpoint: GridRoute NL',
 'status': 'skipped',
 'reason': 'missing ./results_check/grpo_gemma4-e2b_mixed'}

## Phase 2, recipe 3: consistency-reward (one candidate recipe, not this project's headline claim)

Adapts Elhady et al.'s cross-lingual consistency-reward mechanism to cross-format spatial reasoning -- try it, report honestly whether it beats `mixed` or not. See `train_grpo.py`'s `make_reward_fn` docstring for the exact mechanism. This condition generates an extra partner completion per reward call (roughly double the per-step cost of single/mixed) -- `steps_for_budget()` accounts for that automatically since it reads THIS condition's own timing test, not single/mixed's.

In [27]:
SFT_CONSISTENCY_DIR = SFT_MIXED_DIR  # consistency reuses the same mixed-format SFT warm-start
GRPO_CONSISTENCY_TIMING_DIR = "./results_check/grpo_gemma4-e2b_consistency_timing"
run_stage("GRPO timing test: consistency-reward",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "8", "--max_steps", "2", "--max_completion_length", "1024",
           "--output_dir", GRPO_CONSISTENCY_TIMING_DIR],
          required_paths=[SFT_CONSISTENCY_DIR], timeout=TIMEOUT_QUICK)



▶ GRPO timing test: consistency-reward
⏭  SKIPPED -- required input not found: ./results_check/sft_gemma4-e2b_mixed


{'stage': 'GRPO timing test: consistency-reward',
 'status': 'skipped',
 'reason': 'missing ./results_check/sft_gemma4-e2b_mixed'}

In [28]:
GRPO_CONSISTENCY_DIR = "./results_check/grpo_gemma4-e2b_consistency"
consistency_steps = steps_for_budget(GRPO_CONSISTENCY_TIMING_DIR)
run_stage(f"GRPO: consistency-reward full run ({consistency_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "8",
           "--max_steps", str(consistency_steps), "--max_completion_length", "1024", "--output_dir", GRPO_CONSISTENCY_DIR],
          required_paths=[SFT_CONSISTENCY_DIR], timeout=TIMEOUT_GRPO_FULL)


  (no timing.json at ./results_check/grpo_gemma4-e2b_consistency_timing, using default 3 steps)

▶ GRPO: consistency-reward full run (3 steps)
⏭  SKIPPED -- required input not found: ./results_check/sft_gemma4-e2b_mixed


{'stage': 'GRPO: consistency-reward full run (3 steps)',
 'status': 'skipped',
 'reason': 'missing ./results_check/sft_gemma4-e2b_mixed'}

In [29]:
run_stage("Eval consistency-reward checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "mazebench", "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval consistency-reward checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR], timeout=TIMEOUT_EVAL)



▶ Eval consistency-reward checkpoint: MazeBench
⏭  SKIPPED -- required input not found: ./results_check/grpo_gemma4-e2b_consistency

▶ Eval consistency-reward checkpoint: GridRoute NL
⏭  SKIPPED -- required input not found: ./results_check/grpo_gemma4-e2b_consistency


{'stage': 'Eval consistency-reward checkpoint: GridRoute NL',
 'status': 'skipped',
 'reason': 'missing ./results_check/grpo_gemma4-e2b_consistency'}

## Run summary

At-a-glance status of every stage attempted -- read this first when checking back on an unattended run, before digging into the full cell outputs above.

In [30]:
import pandas as pd

summary_df = pd.DataFrame(stage_results)
if not summary_df.empty:
    ok = (summary_df["status"] == "ok").sum()
    print(f"{ok}/{len(summary_df)} stages completed successfully.\n")
pd.set_option("display.max_colwidth", None)
summary_df


6/25 stages completed successfully.



,stage,status,elapsed_min,exit_code,reason
0,Feasibility check (all candidate models),ok,3.8,NaN,NaN
1,"AlphaMaze baseline: MazeBench (full precision, pipeline check)",ok,1.2,NaN,NaN
2,AlphaMaze baseline: GridRoute NL,ok,0.9,NaN,NaN
3,Gemma 4 E2B baseline: MazeBench,timeout,15.0,NaN,NaN
4,Gemma 4 E2B baseline: GridRoute NL,ok,5.9,NaN,NaN
5,Gemma 4 E4B baseline: MazeBench,timeout,15.0,NaN,NaN
6,Gemma 4 E4B baseline: GridRoute NL,timeout,15.0,NaN,NaN
7,[full-precision] Gemma 4 E2B baseline: MazeBench,ok,12.2,NaN,NaN
8,[full-precision] Gemma 4 E2B baseline: GridRoute NL,ok,7.3,NaN,NaN
9,[full-precision] Gemma 4 E4B baseline: MazeBench,failed,0.5,1.0,NaN


## Pipeline check verdict

`run_stage()` deliberately never raises -- that's what lets an unattended multi-hour real run survive one bad stage. But this notebook exists to answer one question (did everything work?), so this cell inverts that: it fails loudly, on purpose, the moment anything didn't complete cleanly.

In [31]:
failed = [r for r in stage_results if r["status"] != "ok"]

print(f"\n{'='*70}")
if failed:
    print(f"CHECK FAILED -- {len(failed)}/{len(stage_results)} stage(s) did not complete cleanly:\n")
    for r in failed:
        detail = r.get("error") or (f"exit code {r['exit_code']}" if "exit_code" in r else r.get("reason", ""))
        print(f"  [{r['status'].upper()}] {r['stage']}" + (f" -- {detail}" if detail else ""))
    print(f"{'='*70}")
    raise RuntimeError(
        f"{len(failed)} pipeline stage(s) failed or were skipped -- see the list above and the "
        "full cell output higher in this notebook for the actual error/traceback. Fix before "
        "trusting a real kaggle_train.ipynb run."
    )
else:
    print(f"CHECK PASSED -- all {len(stage_results)} attempted stages completed successfully.")
    print("Every stage of the real pipeline (setup, feasibility, Phase 1, Phase 1b, "
          "Phase 2 recipes 1-3, eval) ran end to end on this hardware without error.")
    print(f"{'='*70}")



CHECK FAILED -- 19/25 stage(s) did not complete cleanly:

  [TIMEOUT] Gemma 4 E2B baseline: MazeBench
  [TIMEOUT] Gemma 4 E4B baseline: MazeBench
  [TIMEOUT] Gemma 4 E4B baseline: GridRoute NL
  [FAILED] [full-precision] Gemma 4 E4B baseline: MazeBench -- exit code 1
  [FAILED] [full-precision] Gemma 4 E4B baseline: GridRoute NL -- exit code 1
  [FAILED] SFT: single-format (Gemma 4 E2B) -- exit code 1
  [SKIPPED] GRPO timing test: single-format -- missing ./results_check/sft_gemma4-e2b_single
  [SKIPPED] GRPO: single-format full run (3 steps) -- missing ./results_check/sft_gemma4-e2b_single
  [SKIPPED] Eval single-format checkpoint: MazeBench -- missing ./results_check/grpo_gemma4-e2b_single
  [SKIPPED] Eval single-format checkpoint: GridRoute NL -- missing ./results_check/grpo_gemma4-e2b_single
  [FAILED] SFT: mixed-format (Gemma 4 E2B) -- exit code 1
  [SKIPPED] GRPO timing test: mixed-format -- missing ./results_check/sft_gemma4-e2b_mixed
  [SKIPPED] GRPO: mixed-format full run (3 

RuntimeError: 19 pipeline stage(s) failed or were skipped -- see the list above and the full cell output higher in this notebook for the actual error/traceback. Fix before trusting a real kaggle_train.ipynb run.

## Aggregate benchmark results into one comparison table

In [ ]:
import glob

rows = []
for path in sorted(glob.glob("./results_check/eval/*.json")):
    try:
        with open(path) as f:
            d = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        print(f"Skipping unreadable results file {path}: {e}")
        continue
    label = path.split("/")[-1]
    row = {"file": label, "model": d.get("model"), "checkpoint": d.get("checkpoint") or "(none)",
           "benchmark": d.get("benchmark"), "n": d.get("n"),
           "4bit": d.get("load_in_4bit")}  # AlphaMaze rows are always False (enforced in eval.py);
                                            # False Gemma 4 rows are the Phase 1b ablation
    if "mazebench" in d.get("benchmark", ""):
        row["score"] = d.get("accuracy")
        row["used_official_scoring"] = d.get("used_official_scoring")
    else:
        row["valid_rate"] = d.get("valid_rate")
        row["optimal_rate"] = d.get("optimal_rate")
    rows.append(row)

results_df = pd.DataFrame(rows)
if not results_df.empty:
    results_df.to_csv("./results_check/comparison_table.csv", index=False)
else:
    print("No eval result files found yet in ./results_check/eval/ -- nothing to aggregate.")
results_df


## Download results

Small, diagnostic-only -- this is the check run's tiny-sample output, not a substitute for `kaggle_train.ipynb`'s real results.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results_check", "zip", "./results_check")
print("Saved: /kaggle/working/results_check.zip -- download it from the Kaggle output panel on the right.")
print("Includes stage_log.json (this run's full stage-by-stage status) and comparison_table.csv.")
